# Get started with Kumo Relational

Kumo Relational is a foundation model for relational data.
It makes predictions from linked tables, such as customers, products and orders, with no feature engineering and no model to train for each task.
You say what to predict in a short predictive query, and the model answers it from examples it reads from your own tables at the moment you ask.

In this notebook you will:

- deploy the model to a Model Serving endpoint in your workspace;
- describe a set of linked tables to it;
- forecast product demand, find customers who are likely to stop ordering, predict customer spend and large orders, and fill in a missing product attribute;
- predict a measure of a Unity Catalog metric view;
- compare each prediction with what actually happened.

The examples use a sample retail sales dataset: customers, products, orders and order lines, from Microsoft's AdventureWorks.

## How Kumo Relational works

Kumo Relational is built on three ideas.

> **1. Enterprise data is a graph.**

Tables are connected by keys.
In this notebook, each order belongs to a customer, and each order line links a customer to a product.
These links give the model the context around every row: a product's buyers, a customer's past orders, and so on.

> **2. Timestamps place events on a timeline.**

A table whose rows happen at a point in time, such as orders, is an event table.
With events on a timeline, you can pick any moment, the prediction time, and predict what happens next from what came before it.

> **3. Machine learning tasks can be written as predictive queries.**

A predictive query says what to predict, and for which entity, such as each product or each customer.
Forecasting, regression, classification and filling in missing values can all be written this way.
If you know SQL, the Predictive Query Language (PQL) will feel familiar:

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in the next 30 days
FOR EACH products.ProductID                              ← one prediction per product
```

There is no training step.
When you ask for a prediction, the client reads past cases with known outcomes from your tables, called context examples, and the model predicts from them.

## Before you start

You need:

- the Kumo Relational listing installed from Databricks Marketplace, and the name of the catalog you installed it into;
- permission to create a Model Serving endpoint, in a workspace region with GPU Model Serving;
- a catalog and schema where you can create tables, for the sample data;
- serverless compute with environment 4 or later, or Databricks Runtime 17.3 LTS or later.

Time and cost: the first deployment of the endpoint takes 20 to 40 minutes, and the endpoint bills for GPU time while it runs.
It scales to zero when idle, and the last step shows how to delete it.
Once the endpoint is ready, the rest of the notebook runs in a few minutes.

Run the cells in order with Shift+Enter, or use Run all.

**Let's get started!**

## Step 1. Install the client

The Kumo Relational client is the `structured-data-models-client` package.
Install it with the extras for Databricks, then restart Python so the new packages load.
pip may print a red message about `protobuf` versions; it does not affect this notebook.

In [ ]:
%pip install --quiet "structured-data-models-client[relational,databricks,databricks-serving]"
dbutils.library.restartPython()

## Step 2. Set your parameters

Set the names below, then run the cell.
`MODEL_NAME` is the model from the listing, in the catalog you installed it into.
`CATALOG` and `SCHEMA` are where the notebook saves the sample tables.

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)
from sdm_connectors import DatabricksSource

# The model from the listing, as <catalog>.<schema>.<model>, and its version.
MODEL_NAME = 'kumo_relational.models.kumo_relational'
MODEL_VERSION = '1'
ENDPOINT_NAME = 'kumo-relational'
# Where the sample tables go: a catalog and schema you can create tables in.
CATALOG = 'main'
SCHEMA = 'kumo_relational_quickstart'


def full_name(table):
    # The full Unity Catalog name of a sample table, quoted for SQL.
    return f'`{CATALOG}`.`{SCHEMA}`.`adventureworks_{table}`'

## Step 3. Deploy the model

The model runs on a Model Serving endpoint in your own workspace.
The next cell creates the endpoint on one GPU (`GPU_MEDIUM`, size Small), set to scale to zero when idle, or finds it if it already exists.
It does not wait for the deployment, so carry on with the next steps; the notebook waits for the endpoint just before the first prediction.
You can also create the endpoint from the model's page in Catalog Explorer with **Serve this model**, using the same name and settings.

In [ ]:
workspace = WorkspaceClient()
try:
    workspace.serving_endpoints.get(ENDPOINT_NAME)
    print(f'The endpoint {ENDPOINT_NAME} already exists.')
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_type=ServingModelWorkloadType.GPU_MEDIUM,
                    workload_size='Small',
                    scale_to_zero_enabled=True,
                )
            ]
        ),
    )
    print(
        f'Creating the endpoint {ENDPOINT_NAME}; this takes 20 to 40 minutes.'
    )

## Step 4. Load the sample data

This notebook uses four tables from Microsoft's AdventureWorks sample database: customers, products, order headers and order lines.
The next cell downloads them from the [Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script), pinned to one commit so the data cannot change.
The download needs internet access from your compute.
The files are tab-separated with no header row, so the cell names the columns, reads the dates, and copies each order's `CustomerID` and `OrderDate` onto its order lines.
It saves the tables to your schema as `adventureworks_customers`, `adventureworks_products`, `adventureworks_sales_order_headers` and `adventureworks_sales_order_details`, and skips all of this if they are already there.

> Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
> The repository's samples are provided under the [MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

In [ ]:
URL = (
    'https://raw.githubusercontent.com/microsoft/sql-server-samples/'
    'b47eadc85277f72843647223cde53aa1b85ffcb8/samples/databases/'
    'adventure-works/oltp-install-script/{}.csv'
)
# Each table's file, and the file's columns in order.
FILES = {
    'customers': (
        'Customer',
        'CustomerID PersonID StoreID TerritoryID AccountNumber rowguid '
        'ModifiedDate',
    ),
    'products': (
        'Product',
        'ProductID Name ProductNumber MakeFlag FinishedGoodsFlag Color '
        'SafetyStockLevel ReorderPoint StandardCost ListPrice Size '
        'SizeUnitMeasureCode WeightUnitMeasureCode Weight DaysToManufacture '
        'ProductLine Class Style ProductSubcategoryID ProductModelID '
        'SellStartDate SellEndDate DiscontinuedDate rowguid ModifiedDate',
    ),
    'sales_order_headers': (
        'SalesOrderHeader',
        'SalesOrderID RevisionNumber OrderDate DueDate ShipDate Status '
        'OnlineOrderFlag SalesOrderNumber PurchaseOrderNumber AccountNumber '
        'CustomerID SalesPersonID TerritoryID BillToAddressID ShipToAddressID '
        'ShipMethodID CreditCardID CreditCardApprovalCode CurrencyRateID '
        'SubTotal TaxAmt Freight TotalDue Comment rowguid ModifiedDate',
    ),
    'sales_order_details': (
        'SalesOrderDetail',
        'SalesOrderID SalesOrderDetailID CarrierTrackingNumber OrderQty '
        'ProductID SpecialOfferID UnitPrice UnitPriceDiscount LineTotal '
        'rowguid ModifiedDate',
    ),
}

if all(spark.catalog.tableExists(full_name(table)) for table in FILES):
    print(f'The sample tables are already in {CATALOG}.{SCHEMA}.')
else:
    frames = {}
    for table, (file, columns) in FILES.items():
        frame = pd.read_csv(
            URL.format(file),
            sep='\t',
            header=None,
            names=columns.split(),
            dtype_backend='numpy_nullable',
        )
        for column in frame.columns:
            if column.endswith('Date'):
                frame[column] = pd.to_datetime(frame[column])
        frames[table] = frame
    orders = frames['sales_order_headers'][
        ['SalesOrderID', 'CustomerID', 'OrderDate']
    ]
    frames['sales_order_details'] = frames['sales_order_details'].merge(
        orders, on='SalesOrderID', validate='many_to_one'
    )
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`')
    for table, frame in frames.items():
        spark.createDataFrame(frame).write.mode('ignore').saveAsTable(
            full_name(table)
        )
    print(f'Saved the sample tables to {CATALOG}.{SCHEMA}.')

## Step 5. Describe the tables

The model needs each table's primary key, how the tables link, and which tables are event tables.
`DatabricksSource` reads the tables where they are, with your Unity Catalog permissions, and sends the endpoint only the rows each prediction needs.
The sample tables declare no keys in Unity Catalog, so the next cell states them.

- Order headers and order lines are event tables, and `OrderDate` is their time column.
- Customers and products describe things rather than events, so they have no time column.
- Order lines link straight to customers and products rather than through order headers, which keeps each request small.

The cell also leaves out columns the model should not use:

- `rowguid`, a random ID per row, which carries no information;
- `ModifiedDate`, the time a row was last edited, which would show the model data from after the prediction time;
- names, order numbers and account numbers, which are free text, and the model does not read free text.

In [ ]:
source = DatabricksSource(spark, catalog=CATALOG, schema=SCHEMA)
schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_headers': 'SalesOrderID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
        'customers': None,
        'products': None,
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)
leave_out = {
    'customers': ['rowguid', 'ModifiedDate', 'AccountNumber'],
    'products': ['rowguid', 'ModifiedDate', 'Name', 'ProductNumber'],
    'sales_order_headers': [
        'rowguid',
        'ModifiedDate',
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
        'Comment',
    ],
    'sales_order_details': ['rowguid', 'ModifiedDate'],
}
for table, columns in leave_out.items():
    for column in columns:
        del schema[table][column]

print(*schema.tables.values(), sep='\n')
print(*schema.relationships, sep='\n')

## Step 6. Connect to the model

The next cell waits until the endpoint is ready, which can take up to an hour after it was created, and then connects the client to it.
If the endpoint has scaled to zero, the first prediction also takes a few minutes while it starts.

In [ ]:
endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
    ENDPOINT_NAME, timeout=timedelta(hours=1)
)
print(f'The endpoint {ENDPOINT_NAME} is {endpoint.state.ready.value}.')
# Allow 15 minutes per request, for an endpoint that has to scale up from zero.
client = SDMClient.for_databricks_serving(ENDPOINT_NAME, timeout=900)
model = KumoRelational()

## Step 7. Make predictions

Each prediction takes three calls:

1. Write the predictive query with `PredictiveQuery(text, schema)`.
2. Call `materialize(indices=[...])` with the IDs of the entities to predict for. It reads the rows the model needs from your tables, context examples included.
3. Send the request with `client.predict(request, model=model)`, and turn the result into a pandas DataFrame with `to_pandas()`.

**Note:** AdventureWorks is sample data, and the queries and results below are for demonstration.
Benchmark Kumo Relational on data representative of your own use case before drawing conclusions about its performance.

#### Predictive queries in short

Learn more in the [PQL reference](https://docs.nvidia.com/sdgm/reference/pq-reference-overview) and the [guide to writing predictive queries](https://docs.nvidia.com/sdgm/rfm/writing-predictive-queries).

<details>
<summary><b>Click here for a short introduction to predictive queries</b></summary>

**Entities**

`FOR EACH <table>.<primary key>` names the entity, such as `FOR EACH customers.CustomerID`.
You pass the IDs to predict for as `indices` to `materialize`, so the same query works for any set of IDs.

**Targets**

- `PREDICT <table>.<column>` predicts a value of the entity itself, such as a missing attribute.
- `PREDICT <aggregation>(<table>.<column>, <start>, <end>, <unit>)` predicts an aggregate over a time window after the prediction time. `SUM(sales_order_details.OrderQty, 0, 30, days)` is the units sold in the 30 days after it.

**Aggregations**

`COUNT`, `SUM`, `AVG`, `MIN` and `MAX`.
`COUNT(<table>.*, ...)` counts rows, such as orders.

**Conditions**

A comparison after the target turns it into a yes-or-no question: `COUNT(sales_order_headers.*, 0, 90, days) = 0` asks whether a customer places no order in the next 90 days.
A `WHERE` inside the aggregation counts only some rows: `COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, 0, 90, days)` counts only orders over 1,000.
</details>

### Example 1: Forecast product demand

How many units of each of the 10 best-selling products will sell in the next 30 days?

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in the next 30 days
FOR EACH products.ProductID                              ← one prediction per product
```

The cell picks the 10 products that sold the most units in the last 90 days, with SQL.
`num_context_samples=2000` has the model read 2,000 context examples instead of the default 1,000.
More examples from your own history usually give better predictions, at the cost of a larger and slower request, up to 10,000.

In [ ]:
DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)
latest = pd.Timestamp(
    spark.sql(
        f'SELECT max(OrderDate) FROM {full_name("sales_order_headers")}'
    ).first()[0]
)
names = (
    spark.table(full_name('products')).toPandas().set_index('ProductID')['Name']
)
best_sellers = spark.sql(f"""
    SELECT ProductID
    FROM {full_name('sales_order_details')}
    WHERE OrderDate > '{latest - timedelta(days=90)}'
    GROUP BY ProductID
    ORDER BY sum(OrderQty) DESC, ProductID
    LIMIT 10
""").toPandas()

request = PredictiveQuery(DEMAND, schema).materialize(
    indices=best_sellers['ProductID'].tolist(), num_context_samples=2000
)
demand = client.predict(request, model=model).to_pandas()

best_sellers['product'] = best_sellers['ProductID'].map(names)
# A forecast for a product that has stopped selling can dip just below zero.
best_sellers['forecast'] = best_sellers['ProductID'].map(
    demand.set_index('id')['prediction'].clip(lower=0)
)
display(best_sellers.round(1))

How to interpret the result:

1. `ProductID` and `product`: the product each forecast is for.
2. `forecast`: the predicted units ordered in the 30 days after the prediction time, which defaults to the last order in the data.

This result can support inventory and demand planning.
Those 30 days have not happened yet, so this forecast cannot be checked; Example 2 shows how to check one.
The note printed with it says the order headers were not used: from a product, they are further away than the client reads by default, and the order lines already carry each order's customer and date.

### Example 2: Check a forecast against actual sales

How good is the forecast?
To find out, move the prediction time into the past with `anchor_time`.
The model then sees only the data up to that time, so its forecast can be compared with what actually sold afterwards.

Here the prediction time is 180 days before the last order, and the forecast is for the 30 products that sold the most units up to then.
The cell also reads each product's actual revenue in the same 30 days, for Example 7.

In [ ]:
PAST = latest - timedelta(days=180)
NEXT_30_DAYS = (
    f"OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=30)}'"
)
past_best_sellers = spark.sql(f"""
    SELECT ProductID,
           sum(OrderQty) FILTER (WHERE OrderDate <= '{PAST}') AS units_before,
           coalesce(sum(OrderQty) FILTER (WHERE {NEXT_30_DAYS}), 0) AS actual,
           coalesce(sum(LineTotal) FILTER (WHERE {NEXT_30_DAYS}), 0) AS actual_revenue
    FROM {full_name('sales_order_details')}
    GROUP BY ProductID
    ORDER BY units_before DESC NULLS LAST, ProductID
    LIMIT 30
""").toPandas()

request = PredictiveQuery(DEMAND, schema).materialize(
    indices=past_best_sellers['ProductID'].tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
past_demand = client.predict(request, model=model).to_pandas()

past_best_sellers['product'] = past_best_sellers['ProductID'].map(names)
past_best_sellers['forecast'] = past_best_sellers['ProductID'].map(
    past_demand.set_index('id')['prediction'].clip(lower=0)
)
print(f'Prediction time: {PAST.date()}')
display(
    past_best_sellers[['ProductID', 'product', 'forecast', 'actual']].round(1)
)

How to interpret the result:

1. `forecast`: the units the model predicted, at the prediction time, for the next 30 days.
2. `actual`: the units actually ordered in those 30 days.

Most forecasts are close to the actual sales, and products the model predicted near zero, such as the Road-650 bikes, did stop selling.
Step 8 charts all 30 forecasts against the actual sales.
Checking a query this way, at a past prediction time, is how to judge it on your own data before you rely on it.

### Example 3: Find customers who are likely to stop ordering

Which active customers will place no order in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

The `= 0` turns the count into a yes-or-no question, so the result includes a probability.
The cell asks it at the same past prediction time, for a fixed random sample of 200 customers who ordered in the year before it, and reads what each of them actually did in the 90 days after it.
Examples 4 and 5 use the same customers.
The cell reads 5,000 context examples: with the default 1,000, the model tells customers who stop ordering from those who keep ordering much less well.

In [ ]:
NEXT_90_DAYS = (
    f"OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=90)}'"
)
customers = spark.sql(f"""
    SELECT CustomerID,
           count_if({NEXT_90_DAYS}) AS orders,
           coalesce(sum(TotalDue) FILTER (WHERE {NEXT_90_DAYS}), 0) AS spend,
           count_if({NEXT_90_DAYS} AND TotalDue > 1000) AS large_orders
    FROM {full_name('sales_order_headers')}
    GROUP BY CustomerID
    HAVING count_if(OrderDate <= '{PAST}' AND OrderDate > '{PAST - timedelta(days=365)}') > 0
    ORDER BY xxhash64(CustomerID), CustomerID
    LIMIT 200
""").toPandas()


def by_group(result, happened):
    # Four equal groups of customers by predicted probability, each with its
    # average prediction and how often the event actually happened.
    group = pd.qcut(
        result['probability_True'].rank(method='first'),
        4,
        labels=['lowest', 'low', 'high', 'highest'],
    )
    return (
        result.assign(group=group, happened=happened)
        .groupby('group', observed=True)
        .agg(
            customers=('happened', 'size'),
            predicted=('probability_True', 'mean'),
            actual=('happened', 'mean'),
        )
        .round(2)
    )


request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers['CustomerID'].tolist(),
    anchor_time=PAST,
    num_context_samples=5000,
)
churn = client.predict(request, model=model).to_pandas()

churn = churn.merge(customers, left_on='id', right_on='CustomerID')
churn_groups = by_group(churn, churn['orders'] == 0)
display(churn_groups)

How to interpret the result:

1. Each row is a quarter of the 200 customers, grouped by `probability_True`, the predicted probability that the customer places no order.
2. `predicted`: the group's average predicted probability.
3. `actual`: the share of the group that placed no order in the 90 days.

The groups the model rated riskier stopped ordering more often, so the probability can rank customers for retention or re-engagement work.
Most customers in this dataset order only now and then, so even the lowest-risk group has a high churn rate.

### Example 4: Predict each customer's spend

How much will each of the same customers spend in the next 90 days?

```
PREDICT SUM(sales_order_headers.TotalDue, 0, 90, days)   ← order value in the next 90 days
FOR EACH customers.CustomerID                            ← one prediction per customer
```

`prediction_statistic='mean'` makes each prediction the expected spend rather than the median.
Most customers spend nothing in any given 90 days, so the median is usually 0, while expected spend can be added up across customers.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(sales_order_headers.TotalDue, 0, 90, days) '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers['CustomerID'].tolist(),
    anchor_time=PAST,
    num_context_samples=5000,
    prediction_statistic='mean',
)
spend = client.predict(request, model=model).to_pandas()

spend = spend.merge(customers, left_on='id', right_on='CustomerID')
spend['predicted'] = spend['prediction'].clip(lower=0)
top_20 = spend.nlargest(20, 'predicted')
top_20_share = top_20['spend'].sum() / spend['spend'].sum()
print(
    f'The 20 customers with the highest predicted spend made {top_20_share:.0%} '
    f'of the actual spend of all 200.'
)
display(top_20.head(10)[['CustomerID', 'predicted', 'spend']].round(0))

How to interpret the result:

1. `predicted`: the customer's expected spend in the 90 days after the prediction time.
2. `spend`: what the customer actually spent in those 90 days.

The table lists the 10 customers with the highest predicted spend.
A random 20 of the 200 customers would account for about a tenth of the spend, so the share printed above shows how well the prediction finds the customers who matter most.

### Example 5: Predict who will place a large order

Which of the same customers will place an order over 1,000 in the next 90 days?

```
PREDICT COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, 0, 90, days) > 0
FOR EACH customers.CustomerID
```

The `WHERE` inside the aggregation counts only large orders, and `> 0` asks whether there is at least one.

In [ ]:
request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, '
    '0, 90, days) > 0 FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers['CustomerID'].tolist(),
    anchor_time=PAST,
    num_context_samples=5000,
)
large = client.predict(request, model=model).to_pandas()

large = large.merge(customers, left_on='id', right_on='CustomerID')
large_groups = by_group(large, large['large_orders'] > 0)
display(large_groups)

How to interpret the result:

1. Each row is a quarter of the 200 customers, grouped by `probability_True`, the predicted probability of a large order.
2. `actual`: the share of the group that placed a large order in the 90 days.

Large orders are rare, and they came mostly from the group the model rated most likely and not at all from the half it rated least likely.
This makes the probability a way to choose whom to contact for an upsell.

### Example 6: Fill in a missing product attribute

A query with no time window predicts a value of the entity itself, which fills in values that are missing.

```
PREDICT products.ProductLine   ← R road, M mountain, T touring or S standard
FOR EACH products.ProductID    ← one prediction per product
```

To check it, the cell predicts the product line of 60 products picked at random, and compares the predictions with the recorded values.
A product's own recorded value is never among the context examples, so the model cannot copy it; the client prints a note saying so.

In [ ]:
lines = spark.sql(f"""
    SELECT ProductID, Name AS product, trim(ProductLine) AS recorded
    FROM {full_name('products')}
    WHERE ProductLine IS NOT NULL
    ORDER BY xxhash64(ProductID), ProductID
    LIMIT 60
""").toPandas()

request = PredictiveQuery(
    'PREDICT products.ProductLine FOR EACH products.ProductID', schema
).materialize(indices=lines['ProductID'].tolist())
product_lines = client.predict(request, model=model).to_pandas()

lines['predicted'] = lines['ProductID'].map(
    product_lines.set_index('id')['prediction'].str.strip()
)
correct = (lines['predicted'] == lines['recorded']).sum()
print(f'{correct} of {len(lines)} product lines predicted correctly.')
display(lines[lines['predicted'] != lines['recorded']])

How to interpret the result:

1. The printed line counts the correct predictions; always guessing the most common line would get about a third right.
2. The table lists the products the model got wrong, with the recorded and the predicted line.

Several of the misses have their line only in the product name, such as Touring Tire, and the model does not read free text.
Predicted values like these can fill gaps in a catalog, and should be reviewed before they are written back to a source system.

### Example 7: Predict from a metric view

A Unity Catalog metric view defines how tables join and which measures matter, such as revenue.
Kumo Relational can read that definition, so you can predict a measure without describing the tables yourself.
The next cell creates a metric view over the sample tables in your schema, `adventureworks_sales_metrics`, unless it already exists.

In [ ]:
spark.sql(f"""
CREATE VIEW IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`adventureworks_sales_metrics`
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: "{full_name('sales_order_details')}"
joins:
  - name: customers
    source: "{full_name('customers')}"
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: "{full_name('products')}"
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
  - name: revenue
    expr: SUM(source.LineTotal)
  - name: order_count
    expr: COUNT(DISTINCT source.SalesOrderID)
$$
""")
print(f'The metric view adventureworks_sales_metrics is in {CATALOG}.{SCHEMA}.')

`source.semantic_model` reads the view, and `Schema.from_source` turns it into a schema with its tables, keys and links.
The view's main table keeps its own name, `adventureworks_sales_order_details`, and each joined table takes its join name, such as `products`.
For each measure, the client prints the target that computes it, ready to use in a predictive query.
A distinct count, such as `order_count`, is not one the model can predict.

In [ ]:
view_schema = Schema.from_source(
    source.semantic_model('adventureworks_sales_metrics')
)
print(*view_schema.tables.values(), sep='\n')
print(*view_schema.conversion_notes, sep='\n')

The next cell forecasts the `revenue` measure for the same 30 products and the same past prediction time as Example 2, and shows it next to the actual revenue.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 30, days) '
    'FOR EACH products.ProductID',
    view_schema,
).materialize(
    indices=past_best_sellers['ProductID'].tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
revenue = client.predict(request, model=model).to_pandas()

past_best_sellers['revenue_forecast'] = past_best_sellers['ProductID'].map(
    revenue.set_index('id')['prediction'].clip(lower=0)
)
display(
    past_best_sellers[
        ['ProductID', 'product', 'revenue_forecast', 'actual_revenue']
    ].round(0)
)

How to interpret the result:

1. `revenue_forecast`: the predicted revenue in the 30 days after the prediction time, from the view's `revenue` measure.
2. `actual_revenue`: the revenue actually made in those 30 days.

A team that already keeps its business definitions in metric views can predict those same measures, without restating its tables.

## Step 8. Compare the predictions with what happened

The chart shows each of the 30 products from Example 2 at its actual and its forecast units; points on the gray line are exact.
The table after it sums up every check in this notebook.

In [ ]:
top = 1.05 * past_best_sellers[['forecast', 'actual']].to_numpy().max()
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, top], [0, top], color='#a3a29b', linewidth=1)
ax.scatter(
    past_best_sellers['actual'],
    past_best_sellers['forecast'],
    s=48,
    color='#2a78d6',
    edgecolor='white',
    linewidth=1,
    zorder=3,
)
ax.set(
    xlim=(0, top), ylim=(0, top), xlabel='Actual units', ylabel='Forecast units'
)
ax.set_title(f'The 30 best sellers, 30 days after {PAST.date()}', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.show()


def median_error(forecast, actual):
    # The typical forecast error, as a share of the actual value, over the
    # products that sold anything in the 30 days.
    sold = past_best_sellers[past_best_sellers[actual] > 0]
    return ((sold[forecast] - sold[actual]).abs() / sold[actual]).median()


summary = pd.DataFrame(
    [
        (
            '2. Product demand',
            'Median forecast error, products that sold',
            f'{median_error("forecast", "actual"):.0%}',
        ),
        (
            '3. Customers who stop ordering',
            'Share that placed no order: lowest-risk group, highest-risk group',
            f'{churn_groups["actual"].iloc[0]:.0%}, {churn_groups["actual"].iloc[-1]:.0%}',
        ),
        (
            '4. Customer spend',
            'Share of actual spend from the 20 highest predictions (random: about 10%)',
            f'{top_20_share:.0%}',
        ),
        (
            '5. Large orders',
            'Share that placed one: most likely group, the other three groups',
            f'{large_groups["actual"].iloc[-1]:.0%}, '
            f'{large_groups["actual"].iloc[:-1].mean():.0%}',
        ),
        (
            '6. Product line',
            'Correct predictions (most common line: about a third)',
            f'{correct} of {len(lines)}',
        ),
        (
            '7. Revenue from a metric view',
            'Median forecast error, products that sold',
            f'{median_error("revenue_forecast", "actual_revenue"):.0%}',
        ),
    ],
    columns=['Example', 'Check', 'Result'],
)
display(summary)

## Step 9. Try it yourself

The next cell repeats the forecast of Example 2 for the first 10 of its products, and shows it next to the actual values.
Change `DAYS` or `COLUMN`, or uncomment one of the challenge lines, and run the cell again.

In [ ]:
DAYS = 30
COLUMN = 'OrderQty'
# Challenge: uncomment a line, run the cell again, and compare with the actuals.
# DAYS = 90  # forecast the next 90 days instead of 30
# COLUMN = 'LineTotal'  # forecast revenue instead of units

request = PredictiveQuery(
    f'PREDICT SUM(sales_order_details.{COLUMN}, 0, {DAYS}, days) '
    'FOR EACH products.ProductID',
    schema,
).materialize(
    indices=past_best_sellers['ProductID'].head(10).tolist(),
    anchor_time=PAST,
    num_context_samples=2000,
)
result = client.predict(request, model=model).to_pandas()

actual = spark.sql(f"""
    SELECT ProductID AS id, sum({COLUMN}) AS actual
    FROM {full_name('sales_order_details')}
    WHERE OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=DAYS)}'
    GROUP BY ProductID
""").toPandas()
result = result.merge(actual, on='id', how='left').fillna({'actual': 0})
result['product'] = result['id'].map(names)
result['prediction'] = result['prediction'].clip(lower=0)
display(result[['id', 'product', 'prediction', 'actual']].round(1))

## Step 10. Use your own data

To use Kumo Relational on your own tables, point a source at your schema, describe the tables, and write a predictive query for your question:

```python
source = DatabricksSource(spark, catalog='<your catalog>', schema='<your schema>')
schema = Schema.from_source(
    source,
    {'users': '<users table>', 'orders': '<orders table>'},
    primary_keys={'users': 'user_id', 'orders': 'order_id'},
    time_columns={'orders': 'ordered_at', 'users': None},
    relationships=[Relationship('orders', 'user_id', 'users', 'user_id')],
)
request = PredictiveQuery(
    'PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id', schema
).materialize(indices=[...], num_context_samples=5000)
result = client.predict(request, model=model).to_pandas()
```

| Question | Predictive query |
|---|---|
| Will this customer stop ordering? | `PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id` |
| How much will this customer spend? | `PREDICT SUM(orders.amount, 0, 90, days) FOR EACH users.user_id` |
| Will this customer place a large order? | `PREDICT COUNT(orders.* WHERE orders.amount > 1000, 0, 90, days) > 0 FOR EACH users.user_id` |
| How many units will this item sell? | `PREDICT SUM(order_lines.quantity, 0, 30, days) FOR EACH items.item_id` |
| What is this item's missing category? | `PREDICT items.category FOR EACH items.item_id` |

Tips:

- Name the time column of each event table, and pass `None` for tables that describe things.
- Leave out columns that record later edits, random row IDs, and free text.
- Before you rely on a query, run it at a past prediction time with `anchor_time` and compare it with what happened, as in Example 2.
- To keep the predictions, save them as a table: `spark.createDataFrame(result).write.saveAsTable('<table>')`.

Learn more:

- [Predictive Query reference](https://docs.nvidia.com/sdgm/reference/pq-reference-overview)
- [Writing predictive queries](https://docs.nvidia.com/sdgm/rfm/writing-predictive-queries)

## Step 11. Clean up

The endpoint bills for GPU time only while it runs, and it scales to zero when idle.
To delete it, uncomment the last line and run the cell.
The sample tables and the metric view stay in your schema until you drop them.

In [ ]:
client.close()
# workspace.serving_endpoints.delete(ENDPOINT_NAME)